# 08 — PSO Ensemble (KAGGLE GPU)
## WavSent-MTL · Tasks 3.17–3.23

**Runs on: Kaggle T4 x2 GPU**

**What this notebook does (Config G):**
- Load saved best-seed val+test predictions from notebooks 06 and 07
- Run PSO weight search on VALIDATION predictions only (not training)
- Apply learned weights to test predictions → Config G metrics
- Done separately for Kotekar and Kaggle datasets

**CRITICAL:** Individual model test metrics must be already saved (notebooks 06/07)  
**PSO uses val predictions only** — this is the key improvement over Kotekar et al.

**PREREQUISITE:** Notebooks 06 and 07 complete, all val_predictions/*.npy uploaded

In [1]:
# ── Kaggle Setup ────────────────────────────────────────────────────────────
import subprocess
import sys
import os

repo_path = '/kaggle/working/WavSent-MTL'

if os.path.exists(repo_path):
    subprocess.run(['git', '-C', repo_path, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone',
        'https://github.com/IAMNEERAJ05/WavSent-MTL.git',
        repo_path], check=True)

sys.path.insert(0, repo_path)
from config.config import CONFIG
print(f"Ready. BEST_REPR: {CONFIG['BEST_REPR']}")

import sys
import os
sys.path.insert(0, '/kaggle/working/WavSent-MTL')
os.chdir('/kaggle/working/WavSent-MTL')

import numpy as np
import pandas as pd
import json
import torch
from config.config import CONFIG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'BEST_REPR: {CONFIG["BEST_REPR"]}')

From https://github.com/IAMNEERAJ05/WavSent-MTL
   33fdb03..7326a9a  main       -> origin/main


Updating 33fdb03..7326a9a
Fast-forward
 notebooks/06_training_kotekar.ipynb | 27 +--------------------------
 notebooks/07_training_kaggle.ipynb  | 24 ++----------------------
 src/training/trainer.py             | 24 ++++++++++++++++++++----
 3 files changed, 23 insertions(+), 52 deletions(-)
Ready. BEST_REPR: denoised_technicals
Device: cuda
BEST_REPR: denoised_technicals


In [2]:
!pip install pyswarms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.0 MB/s eta 0:00:00


## Copy val_predictions from Kaggle Dataset Inputs

> Upload `ablation/results/kotekar/val_predictions/` and `ablation/results/kaggle/val_predictions/`  
> as part of the Kaggle dataset, OR copy from notebook 06/07 working outputs.

In [3]:
# If val_predictions are in the Kaggle dataset input:
KOTEKAR_PREDS_INPUT = '/kaggle/input/datasets/consistency23/wavsent-kotekar-preds'
KAGGLE_PREDS_INPUT  = '/kaggle/input/datasets/consistency23/wavsent-kaggle-preds'

# Copy val prediction files to expected ablation/results/ paths
import shutil

for dataset, src_dir in [('kotekar', KOTEKAR_PREDS_INPUT),
                          ('kaggle',  KAGGLE_PREDS_INPUT)]:
    dst_dir = f'ablation/results/{dataset}'
    os.makedirs(dst_dir, exist_ok=True)
    for model in CONFIG['pso_models']:
        for pred_type in ['val_preds', 'test_preds']:
            fname = f'{model}_{pred_type}.npy'
            src = os.path.join(src_dir, fname)
            dst = os.path.join(dst_dir, fname)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                print(f'Copied: {src} -> {dst}')
            else:
                print(f'NOT FOUND: {src}  (check dataset upload)')

Copied: /kaggle/input/datasets/consistency23/wavsent-kotekar-preds/tkan_val_preds.npy -> ablation/results/kotekar/tkan_val_preds.npy
Copied: /kaggle/input/datasets/consistency23/wavsent-kotekar-preds/tkan_test_preds.npy -> ablation/results/kotekar/tkan_test_preds.npy
Copied: /kaggle/input/datasets/consistency23/wavsent-kotekar-preds/lstm_val_preds.npy -> ablation/results/kotekar/lstm_val_preds.npy
Copied: /kaggle/input/datasets/consistency23/wavsent-kotekar-preds/lstm_test_preds.npy -> ablation/results/kotekar/lstm_test_preds.npy
Copied: /kaggle/input/datasets/consistency23/wavsent-kotekar-preds/gru_val_preds.npy -> ablation/results/kotekar/gru_val_preds.npy
Copied: /kaggle/input/datasets/consistency23/wavsent-kotekar-preds/gru_test_preds.npy -> ablation/results/kotekar/gru_test_preds.npy
Copied: /kaggle/input/datasets/consistency23/wavsent-kotekar-preds/tcn_val_preds.npy -> ablation/results/kotekar/tcn_val_preds.npy
Copied: /kaggle/input/datasets/consistency23/wavsent-kotekar-preds/tc

## PSO Helper

In [4]:
from src.ensemble.pso_ensemble import (
    collect_val_predictions,
    run_pso_search,
    apply_ensemble_weights,
)
from src.evaluation.metrics import compute_clf_metrics


def run_ensemble_for_dataset(dataset, y_clf_val, y_clf_test):
    """Run full PSO pipeline for one dataset."""
    print(f'\n{"=" * 60}')
    print(f'PSO Ensemble: {dataset.upper()}')
    print(f'{"=" * 60}')

    # Load saved val + test predictions
    val_preds, test_preds = collect_val_predictions(dataset=dataset)

    print('Loaded val predictions:')
    for m, p in val_preds.items():
        print(f'  {m}: shape={p.shape}  mean={p.mean():.4f}')

    # Individual model test metrics (stored BEFORE ensemble)
    print('\nIndividual model test metrics (before ensemble):')
    individual_metrics = {}
    for m in CONFIG['pso_models']:
        metrics = compute_clf_metrics(y_clf_test, test_preds[m])
        individual_metrics[m] = metrics
        print(f'  {m}: acc={metrics["accuracy"]:.4f}  auc={metrics["auc"]:.4f}')

    # PSO weight search on validation predictions
    print('\nRunning PSO weight search...')
    weights = run_pso_search(
        val_preds=val_preds,
        val_labels=y_clf_val,
    )

    # Save PSO weights
    weights_path = f'/kaggle/working/pso_weights_{dataset}.json'
    with open(weights_path, 'w') as f:
        json.dump(weights, f, indent=2)
    print(f'PSO weights saved: {weights_path}')

    # Apply to test predictions → Config G
    print('\nConfig G (ensemble) test metrics:')
    g_metrics = apply_ensemble_weights(
        weights=weights,
        test_preds=test_preds,
        test_labels=y_clf_test,
    )

    # Save Config G result
    g_row = {
        'config': 'G', 'model': 'ensemble', 'seed': 0, 'run': 0,
        'dataset': dataset, **g_metrics,
        'val_accuracy': -1,  # PSO val acc stored separately
        'rmse': None, 'mae': None, 'r2': None,
    }
    result_path = f'/kaggle/working/ensemble_results_{dataset}.csv'
    pd.DataFrame([g_row]).to_csv(result_path, index=False)
    print(f'Ensemble results saved: {result_path}')

    return weights, g_metrics, individual_metrics

## Tasks 3.17–3.19 — Kotekar PSO Ensemble

In [5]:
KOTEKAR_INPUT = '/kaggle/input/datasets/consistency23/wavsent-kotekar-processed'
y_clf_val_kot  = np.load(f'{KOTEKAR_INPUT}/y_clf_val.npy')
y_clf_test_kot = np.load(f'{KOTEKAR_INPUT}/y_clf_test.npy')

kot_weights, kot_g_metrics, kot_individual = run_ensemble_for_dataset(
    dataset='kotekar',
    y_clf_val=y_clf_val_kot,
    y_clf_test=y_clf_test_kot,
)

2026-03-23 09:09:20,157 - pyswarms.single.global_best - INFO - Optimize for 50 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}



PSO Ensemble: KOTEKAR
Loaded val predictions:
  tkan: shape=(155,)  mean=0.5109
  lstm: shape=(155,)  mean=0.5373
  gru: shape=(155,)  mean=0.5120
  tcn: shape=(155,)  mean=0.5389

Individual model test metrics (before ensemble):
  tkan: acc=0.5897  auc=0.6083
  lstm: acc=0.5897  auc=0.6606
  gru: acc=0.5962  auc=0.6639
  tcn: acc=0.5833  auc=0.6272

Running PSO weight search...


pyswarms.single.global_best: 100%|██████████|50/50, best_cost=-0.858
2026-03-23 09:09:27,162 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.8580645161290322, best pos: [0.81929788 1.18441356 0.98586371 3.66325425]


PSO best val accuracy: 0.8581
PSO weights: {'tkan': 0.04806412644284751, 'lstm': 0.06924505742962211, 'gru': 0.05677533092241116, 'tcn': 0.8259154852051193}
PSO weights saved: /kaggle/working/pso_weights_kotekar.json

Config G (ensemble) test metrics:
Config G (ensemble) test metrics: {'accuracy': 0.5641025641025641, 'balanced_accuracy': 0.5701871657754011, 'auc': 0.6345254010695187, 'precision': 0.6388888888888888, 'recall': 0.5227272727272727, 'f1': 0.575}
Ensemble results saved: /kaggle/working/ensemble_results_kotekar.csv


## Tasks 3.20–3.21 — Kaggle PSO Ensemble

In [6]:
KAGGLE_INPUT  = '/kaggle/input/datasets/consistency23/wavsent-kaggle-processed'
y_clf_val_kag  = np.load(f'{KAGGLE_INPUT}/y_clf_val.npy')
y_clf_test_kag = np.load(f'{KAGGLE_INPUT}/y_clf_test.npy')

kag_weights, kag_g_metrics, kag_individual = run_ensemble_for_dataset(
    dataset='kaggle',
    y_clf_val=y_clf_val_kag,
    y_clf_test=y_clf_test_kag,
)

2026-03-23 09:09:27,230 - pyswarms.single.global_best - INFO - Optimize for 50 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}



PSO Ensemble: KAGGLE
Loaded val predictions:
  tkan: shape=(264,)  mean=0.5610
  lstm: shape=(264,)  mean=0.5274
  gru: shape=(264,)  mean=0.5252
  tcn: shape=(264,)  mean=0.5165

Individual model test metrics (before ensemble):
  tkan: acc=0.6981  auc=0.7205
  lstm: acc=0.6943  auc=0.7277
  gru: acc=0.6755  auc=0.7326
  tcn: acc=0.5849  auc=0.6854

Running PSO weight search...


pyswarms.single.global_best: 100%|██████████|50/50, best_cost=-0.75 
2026-03-23 09:09:34,252 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.75, best pos: [2.54961084 8.64641037 3.69896808 0.84166715]


PSO best val accuracy: 0.7500
PSO weights: {'tkan': 0.002228310482359215, 'lstm': 0.9903349223473572, 'gru': 0.007032912434717401, 'tcn': 0.0004038547355659393}
PSO weights saved: /kaggle/working/pso_weights_kaggle.json

Config G (ensemble) test metrics:
Config G (ensemble) test metrics: {'accuracy': 0.690566037735849, 'balanced_accuracy': 0.6877144209156512, 'auc': 0.7275523482787176, 'precision': 0.7602739726027398, 'recall': 0.7025316455696202, 'f1': 0.7302631578947368}
Ensemble results saved: /kaggle/working/ensemble_results_kaggle.csv


## Summary

In [7]:
print('=' * 60)
print('Notebook 08 -- PSO Ensemble: COMPLETE')
print('=' * 60)

for dataset, weights, g_metrics, individual in [
    ('kotekar', kot_weights, kot_g_metrics, kot_individual),
    ('kaggle',  kag_weights, kag_g_metrics, kag_individual),
]:
    print(f'\n--- {dataset.upper()} ---')
    print(f'PSO weights: {weights}')
    print(f'Config G accuracy: {g_metrics["accuracy"]:.4f}')
    print(f'Config G AUC:      {g_metrics["auc"]:.4f}')
    print(f'Individual models:')
    for m, m_metrics in individual.items():
        print(f'  {m}: acc={m_metrics["accuracy"]:.4f}')

print()
print('Benchmark (Kotekar et al.): accuracy=0.5853, Sharpe=1.5679')
print(f'Our Config G (kotekar):    accuracy={kot_g_metrics["accuracy"]:.4f}')

print()
print('Download from /kaggle/working/:')
print('  pso_weights_kotekar.json')
print('  pso_weights_kaggle.json')
print('  ensemble_results_kotekar.csv')
print('  ensemble_results_kaggle.csv')
print()
print('Next: run notebook 09_evaluation (LOCAL)')

Notebook 08 -- PSO Ensemble: COMPLETE

--- KOTEKAR ---
PSO weights: {'tkan': 0.04806412644284751, 'lstm': 0.06924505742962211, 'gru': 0.05677533092241116, 'tcn': 0.8259154852051193}
Config G accuracy: 0.5641
Config G AUC:      0.6345
Individual models:
  tkan: acc=0.5897
  lstm: acc=0.5897
  gru: acc=0.5962
  tcn: acc=0.5833

--- KAGGLE ---
PSO weights: {'tkan': 0.002228310482359215, 'lstm': 0.9903349223473572, 'gru': 0.007032912434717401, 'tcn': 0.0004038547355659393}
Config G accuracy: 0.6906
Config G AUC:      0.7276
Individual models:
  tkan: acc=0.6981
  lstm: acc=0.6943
  gru: acc=0.6755
  tcn: acc=0.5849

Benchmark (Kotekar et al.): accuracy=0.5853, Sharpe=1.5679
Our Config G (kotekar):    accuracy=0.5641

Download from /kaggle/working/:
  pso_weights_kotekar.json
  pso_weights_kaggle.json
  ensemble_results_kotekar.csv
  ensemble_results_kaggle.csv

Next: run notebook 09_evaluation (LOCAL)
